# Notebook 03 — controlled patient environment (E-07, E-12, V-01/02/04)

**Scope.** This notebook runs the existing pure disclosure engine on scripted *student* turns. It shows how facts open, how trust and memory change, and what a student can see. Scripted intent tags are inputs to an engineering probe; they are **not** reviewed labels, a trained extractor, or evidence of learning.

**Sources.** [`PRD.md`](../../PRD.md) §§9.1, 9.3, 9.4 and §15; [`AGENTS.md`](../../AGENTS.md); [`cases/walter_greene.json`](../../cases/walter_greene.json); [`cases/alina_reyes.json`](../../cases/alina_reyes.json); [`backend_kelsch/docs/validation/action_data.json`](../docs/validation/action_data.json). The two JSON cases are authored synthetic examples. Their SHA-256 hashes and each released fact's `source` travel with the trace. The external Postzeun dataset remains blocked for import by its unknown license/provenance.

Run from repository root with `LLM_ENABLED=false`; no cloud credentials or external patient data are required. The local text/script interface is for probes, not a student-facing typed encounter.

In [ ]:
from pathlib import Path
import json, os, sys
os.environ["LLM_ENABLED"] = "false"
sys.path.insert(0, str(Path("backend").resolve()))
from app.schemas.case import Case
from backend_kelsch.app.patient_environment.runner import (case_source, fact_for_role, fingerprint,
    replay, run_script, tagged_turn, with_probe_preset)
from backend_kelsch.scripts.eval_patient_environment import scenario, evaluate
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
ROOT = Path.cwd()
assert (ROOT / "cases/walter_greene.json").exists(), "Run from repository root"
case_path = ROOT / "cases/walter_greene.json"
case = Case.model_validate_json(case_path.read_text())
source = case_source(case_path)
source["path"] = str(case_path.relative_to(ROOT))
print(source, "case version", case.version)

## 1. Authored script and provenance

The same six turns are sent to every probe preset. Topics and empathy are explicitly supplied as `Intent` hits with exact student spans. This isolates the engine: Notebook 02 could not validate a learned action classifier. The student first asks about fear at trust 0, then introduces themself and makes two supportive comments. Later questions revisit the fear topic. Walter's `f_failed_family` is reluctant (threshold 3 in the case); `f_scared_future` is hidden (threshold 5). These are case rules, not claims that a patient behaves this way.

In [ ]:
turns = scenario()
for i, item in enumerate(turns, 1):
    print(i, item.utterance, "→", item.intent.topic_ids(), [x.id for x in item.intent.empathy])
for fid in ("f_failed_family", "f_scared_future"):
    fact = case.fact(fid)
    print(fid, fact.tier, fact.unlock.model_dump(), fact.source.model_dump())

## 2. Disclosure, trust and memory trace

The engine applies a trust event before the disclosure check on each turn. It records `fact_id`, topic condition and trust value for each release. A repeated question may cite a fact already disclosed, but it must not create a second disclosure entry. The trace includes the deterministic fallback reply; full semantic checks of LLM phrasing are outside this offline run.

In [ ]:
trace = run_script(case, turns, source=source)
for row in trace["turns"]:
    print(f"turn {row['turn']}: trust {row['trust_before']}→{row['trust_after']}; "
          f"new={row['allowed_fact_ids']}; withheld={[w['fact_id'] for w in row['withheld']]}; "
          f"repeat={row['is_repeat']}")
    print("  evidence:", [(e["event"], e["span"]) for e in row["trust_events"]])
    print("  patient:", row["reply"])
    print("  cited sources:", row["fact_sources"])
assert "f_failed_family" not in trace["turns"][0]["allowed_fact_ids"]
assert "f_failed_family" in trace["turns"][4]["allowed_fact_ids"]
assert trace["turns"][5]["is_repeat"]

### Visual state trace

Upper panel: trust after each turn, with the two guarded thresholds from the case. Lower panel: first disclosure turn for each guarded fact (`—` means still withheld). A line is an authored sequence, not a performance estimate.

In [ ]:
rows = trace["turns"]
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
xs = [r["turn"] for r in rows]
ax.step([0]+xs, [case.trust.start]+[r["trust_after"] for r in rows], where="post", color="#246c9c", label="Trust")
for name, tier, color in [("Reluctant", "reluctant", "#e09c36"), ("Hidden", "hidden", "#ad4a65")]:
    ax.axhline(case.trust.thresholds[tier], linestyle="--", color=color, label=name+" threshold")
ax.set(ylabel="Trust", title="Walter: authored trust and disclosure sequence")
ax.legend(loc="upper left")
for i, fid in enumerate(("f_failed_family", "f_scared_future")):
    first = next((r["turn"] for r in rows if fid in r["allowed_fact_ids"]), None)
    ax2.hlines(i, 0, len(rows), colors="#d3dae2")
    if first:
        ax2.scatter(first, i, s=100, color="#226e53")
        ax2.text(first + .12, i, f"released t{first}", va="center")
    else:
        ax2.text(len(rows)-.7, i, "withheld", va="center")
ax2.set(yticks=[0,1], yticklabels=["Reluctant", "Hidden"], xlim=(0, len(rows)+.7), xlabel="Turn")
ax2.invert_yaxis(); fig.tight_layout(); plt.show()

## 3. Student visibility and adversarial requests

The local `student_view` contains only disclosed fact IDs and released chart entries. Direct lookup of a hidden fact before its disclosure raises `PermissionError`. This proves the wrapper's visibility rule, not authorization of production API endpoints. A numeric invention and an unpermitted citation are rejected by the deterministic verifier. It does **not** establish semantic support for arbitrary prose: that requires a separately evaluated judge and challenge set.

In [ ]:
from app.encounter.verifier import deterministic_checks
first_state = rows[0]["state_after"]
try:
    fact_for_role(case, first_state, "f_scared_future", "student")
except PermissionError as exc:
    print("Student hidden lookup denied:", exc)
print("Student view:", rows[0]["student_view"]["disclosed_fact_ids"])
print("Invented value:", deterministic_checks(case, "My A1C is 9.4.", ["f_soda"], ["f_soda"], [], "What about soda?"))
print("Hidden citation:", deterministic_checks(case, "I'm scared.", ["f_scared_future"], [], [], "How are you?"))

## 4. Difficulty vectors and deterministic replay

These three vectors change **only** case tier thresholds by −2, 0, +2. They are illustrative engineering presets, not faculty-approved difficulty settings. Explicit `min_trust` on individual facts remains authoritative; health literacy, red herrings, and tangentiality are unchanged. The same intents and utterances reproduce the complete state and event ledger bit for bit for a given case version. A different case or modified fact set must not be treated as the same replay.

In [ ]:
preset_traces = {p: run_script(with_probe_preset(case, p), turns, source=source)
                 for p in ("straightforward", "typical", "difficult")}
for preset, t in preset_traces.items():
    cloned = with_probe_preset(case, preset)
    assert fingerprint(t) == fingerprint(replay(t, cloned))
    print(preset, cloned.trust.thresholds, "discovered", len(t["final_state"]["disclosed"]), fingerprint(t)[:16])
fig, ax = plt.subplots(figsize=(8, 3.5))
for preset, t in preset_traces.items():
    ax.step([0]+xs, [0]+[len(r["student_view"]["disclosed_fact_ids"]) for r in t["turns"]],
            where="post", label=preset)
ax.set(xlabel="Turn", ylabel="Disclosed facts", title="Illustrative threshold vectors on one authored script")
ax.legend(); fig.tight_layout(); plt.show()

## 5. Second case: prerequisite and sensitive screening

In the authored Alina case, `f_passive_si` requires trust 1 and `f_no_plan` requires `f_passive_si`. Asking about suicidal ideation alone does not deduct trust. A named introduction supplies the trust event; a later topic question can open the first fact, and another can open its dependent fact. See the case source for the exact authored content and status.

In [ ]:
alina_path = ROOT / "cases/alina_reyes.json"
alina = Case.model_validate_json(alina_path.read_text())
si_turns = [tagged_turn("Have you had thoughts of ending your life?", topics=("suicidal_ideation",)),
    tagged_turn("Hi, I'm Sam, a medical student.", introduction=True),
    tagged_turn("Have you had thoughts of ending your life?", topics=("suicidal_ideation",)),
    tagged_turn("Have you thought about how you would end your life, or made a plan?", topics=("suicidal_ideation",))]
si = run_script(alina, si_turns, source=case_source(alina_path))
for r in si["turns"]:
    print(r["turn"], r["trust_after"], r["allowed_fact_ids"], [w["fact_id"] for w in r["withheld"]])
assert not any(e["delta"] < 0 for e in si["turns"][0]["trust_events"])
assert "f_no_plan" in si["turns"][3]["allowed_fact_ids"]

## 6. Gate to Notebook 04

Run `LLM_ENABLED=false PYTHONPATH=backend python -m backend_kelsch.scripts.eval_patient_environment --output backend_kelsch/docs/validation/patient_environment.json`. The saved report has each check's status and source hash.

| Criterion | Rule | Result here |
|---|---|---|
| Controlled environment | **PASS** when all 10 authored checks pass; **FAIL** on any premature disclosure, visibility leak, unsupported numeric/citation acceptance, transition or replay mismatch | See script result below |
| Trained action extraction | **PASS** only after reviewed, nonleaking train/test encounters satisfy Notebook 02 macro-F1 and critical recall gates; **FAIL** if evaluated below thresholds; **NOT EVALUABLE** without those labels | NOT EVALUABLE |
| Semantic grounding and difficulty calibration | **PASS** with a reviewed challenge set, verified semantic claims and a calibrated multi-student V-04 experiment; **FAIL** on measured safety failure; **NOT EVALUABLE** without evaluation | NOT EVALUABLE |
| Learning effect | **PASS/FAIL** only from an appropriately designed study; otherwise **NOT EVALUABLE** | NOT EVALUABLE |

**Next notebook prompt.** Notebook 04: build scripted novice, competent and expert test students on the three authored cases; record fact-level release expectations independently reviewed by faculty, exercise unsupported semantic claims with the verifier, measure grounded template delivery and disclosure precision/recall, and retain the Notebook 02 trained-model block until licensed and reviewed data are available.

In [ ]:
report = evaluate()
print("Checks", report["passed"], "/", report["total"])
print("Gates:", report["gates"])
assert report["gates"]["controlled_environment"] == "PASS"